In [23]:
import pandas as pd
import numpy as np
import os

# Chemins

DATA_PROCESSED_PATH = "../data/processed"
ML_REPORTS_PATH = "../reports/ml"
EVAL_REPORTS_PATH = "../reports/evaluation"
POWERBI_DATA_PATH = "../data/powerbi"
POWERBI_REPORT_PATH = "../reports/powerbi"

# Création des dossiers

os.makedirs(
    POWERBI_DATA_PATH,
    exist_ok=True
)

os.makedirs(
    POWERBI_REPORT_PATH,
    exist_ok=True
)

print("Dossiers BI prêts")

Dossiers BI prêts


In [24]:
# Données de ventes

dataset_final = pd.read_csv(
    f"{DATA_PROCESSED_PATH}/dataset_final.csv"
)

# Prédictions de test

predictions = pd.read_csv(
    f"{ML_REPORTS_PATH}/predictions_test.csv"
)

# Prévisions futures

future_predictions = pd.read_csv(
    f"{ML_REPORTS_PATH}/previsions_stock_futur.csv"
)

# Résultats des modèles

model_results = pd.read_csv(
    f"{ML_REPORTS_PATH}/model_results.csv"
)

feature_importance = pd.read_csv(
    f"{ML_REPORTS_PATH}/feature_importance.csv"
)

# Résultats d'évaluation

error_by_product = pd.read_csv(
    f"{EVAL_REPORTS_PATH}/error_by_product.csv"
)

error_by_region = pd.read_csv(
    f"{EVAL_REPORTS_PATH}/error_by_region.csv"
)

error_by_country = pd.read_csv(
    f"{EVAL_REPORTS_PATH}/error_by_country.csv"
)

error_by_month = pd.read_csv(
    f"{EVAL_REPORTS_PATH}/error_by_month.csv"
)

evaluation_summary = pd.read_csv(
    f"{EVAL_REPORTS_PATH}/evaluation_summary.csv"
)

# Conversion des dates

dataset_final["date_vente"] = pd.to_datetime(
    dataset_final["date_vente"]
)

dataset_final["date_mois"] = pd.to_datetime(
    dataset_final["date_mois"]
)

predictions["date_mois"] = pd.to_datetime(
    predictions["date_mois"]
)

future_predictions["date_mois"] = pd.to_datetime(
    future_predictions["date_mois"]
)

error_by_month["date"] = pd.to_datetime(
    error_by_month["date"]
)

print("Fichiers chargés")
print("Ventes :", dataset_final.shape)
print("Prédictions de test :", predictions.shape)
print("Prévisions futures :", future_predictions.shape)

Fichiers chargés
Ventes : (28451, 24)
Prédictions de test : (485, 38)
Prévisions futures : (288, 18)


In [25]:
# Agrégation mensuelle des ventes

fact_sales = (
    dataset_final
    .groupby(
        [
            "date_mois",
            "annee",
            "mois",
            "trimestre",
            "categorie_produit_1",
            "categorie_produit_2",
            "produit",
            "region",
            "pays",
            "code_pays",
            "devise"
        ],
        as_index=False
    )
    .agg(
        quantite_vendue=(
            "quantite_vendue",
            "sum"
        ),
        montant_vente=(
            "trans_amount_weight",
            "sum"
        ),
        inflation=(
            "inflation",
            "mean"
        ),
        gdp_growth=(
            "gdp_growth",
            "mean"
        ),
        temperature=(
            "temperature",
            "mean"
        ),
        precipitation=(
            "precipitation",
            "mean"
        ),
        interest_rate=(
            "interest_rate",
            "mean"
        ),
        wheat_price=(
            "wheat_price",
            "mean"
        ),
        corn_price=(
            "corn_price",
            "mean"
        ),
        soybean_price=(
            "soybean_price",
            "mean"
        )
    )
)

print("fact_sales créée")
print("Dimensions :", fact_sales.shape)
print("Pays :", fact_sales["code_pays"].unique())
print("Régions :", fact_sales["region"].unique())

fact_sales.head()

fact_sales créée
Dimensions : (1331, 21)
Pays : ['USA' 'CAN']
Régions : ['Northern America']


,date_mois,annee,mois,trimestre,categorie_produit_1,categorie_produit_2,produit,region,pays,code_pays,...,quantite_vendue,montant_vente,inflation,gdp_growth,temperature,precipitation,interest_rate,wheat_price,corn_price,soybean_price
0,2007-01-01,2007,1,1,agriculture,"cereals, cotton, & feed/forage harvesting",combine harvester (cereals) & cotton picker,Northern America,united states,USA,...,259.0,1572.0,2.852672,2.003858,-2.735484,0.632258,5.320,172.566631,165.160019,255.867379
1,2007-01-01,2007,1,1,agriculture,forestry,feller bunchers (machine),Northern America,united states,USA,...,4.0,16.0,2.852672,2.003858,-2.735484,0.632258,5.320,172.566631,165.160019,255.867379
2,2007-01-01,2007,1,1,agriculture,forestry,"timber handler, log forwarder (machine)",Northern America,canada,CAN,...,4.0,8.0,2.138384,2.049905,-11.700000,0.683871,4.156,172.566631,165.160019,255.867379
3,2007-01-01,2007,1,1,agriculture,soil & crop treatment,sprayer (self propelled),Northern America,united states,USA,...,56.0,290.0,2.852672,2.003858,-2.735484,0.632258,5.320,172.566631,165.160019,255.867379
4,2007-01-01,2007,1,1,agriculture,"wine, fruit, vegetable & cane equipment",straddle (over-the-row) fruit harvester (self ...,Northern America,united states,USA,...,9.0,18.0,2.852672,2.003858,-2.735484,0.632258,5.320,172.566631,165.160019,255.867379


In [26]:
# Colonnes des prédictions historiques

fact_predictions = predictions[
    [
        "date_mois",
        "produit",
        "categorie_produit_1",
        "categorie_produit_2",
        "region",
        "pays",
        "code_pays",
        "quantite_reelle",
        "quantite_predite",
        "erreur",
        "erreur_absolue",
        "modele"
    ]
].copy()

# Variables temporelles

fact_predictions["annee"] = (
    fact_predictions["date_mois"].dt.year
)

fact_predictions["mois"] = (
    fact_predictions["date_mois"].dt.month
)

fact_predictions["trimestre"] = (
    fact_predictions["date_mois"].dt.quarter
)

print("fact_predictions créée")
print("Dimensions :", fact_predictions.shape)

fact_predictions.head()

fact_predictions créée
Dimensions : (485, 15)


,date_mois,produit,categorie_produit_1,categorie_produit_2,region,pays,code_pays,quantite_reelle,quantite_predite,erreur,erreur_absolue,modele,annee,mois,trimestre
0,2015-01-01,aerial cableway,agriculture,forestry,Northern America,canada,CAN,0.0,5.180459,-5.180459,5.180459,Gradient Boosting,2015,1,1
1,2015-01-01,cattle feeder mixer (self propelled),agriculture,livestock,Northern America,canada,CAN,0.0,6.036701,-6.036701,6.036701,Gradient Boosting,2015,1,1
2,2015-01-01,cattle feeder mixer (self propelled),agriculture,livestock,Northern America,united states,USA,0.0,5.631231,-5.631231,5.631231,Gradient Boosting,2015,1,1
3,2015-01-01,combine harvester (cereals) & cotton picker,agriculture,"cereals, cotton, & feed/forage harvesting",Northern America,united states,USA,1239.0,607.804005,631.195995,631.195995,Gradient Boosting,2015,1,1
4,2015-01-01,feller bunchers (machine),agriculture,forestry,Northern America,canada,CAN,0.0,5.180459,-5.180459,5.180459,Gradient Boosting,2015,1,1


In [27]:
# Table des prévisions futures et des stocks recommandés

fact_stock_forecast = future_predictions[
    [
        "date_mois",
        "horizon_mois",
        "produit",
        "categorie_produit_1",
        "categorie_produit_2",
        "region",
        "pays",
        "code_pays",
        "quantite_predite",
        "borne_inferieure",
        "borne_superieure",
        "ecart_type_erreur",
        "stock_securite",
        "point_commande",
        "stock_cible_recommande",
        "niveau_service",
        "delai_approvisionnement_mois",
        "hypothese_open_data"
    ]
].copy()

# Variables temporelles

fact_stock_forecast["annee"] = (
    fact_stock_forecast["date_mois"].dt.year
)

fact_stock_forecast["mois"] = (
    fact_stock_forecast["date_mois"].dt.month
)

fact_stock_forecast["trimestre"] = (
    fact_stock_forecast["date_mois"].dt.quarter
)

print("fact_stock_forecast créée")
print("Dimensions :", fact_stock_forecast.shape)

print(
    "Période :",
    fact_stock_forecast["date_mois"].min(),
    "au",
    fact_stock_forecast["date_mois"].max()
)

fact_stock_forecast.head()

fact_stock_forecast créée
Dimensions : (288, 21)
Période : 2017-01-01 00:00:00 au 2017-12-01 00:00:00


,date_mois,horizon_mois,produit,categorie_produit_1,categorie_produit_2,region,pays,code_pays,quantite_predite,borne_inferieure,...,ecart_type_erreur,stock_securite,point_commande,stock_cible_recommande,niveau_service,delai_approvisionnement_mois,hypothese_open_data,annee,mois,trimestre
0,2017-01-01,1,cattle feeder mixer (self propelled),agriculture,livestock,Northern America,canada,CAN,4.13,0.00,...,8.84,15,20,20,95,1,Valeurs Open Data de décembre 2016 maintenues ...,2017,1,1
1,2017-01-01,1,cattle feeder mixer (self propelled),agriculture,livestock,Northern America,united states,USA,5.52,0.52,...,2.69,5,11,11,95,1,Valeurs Open Data de décembre 2016 maintenues ...,2017,1,1
2,2017-01-01,1,combine harvester (cereals) & cotton picker,agriculture,"cereals, cotton, & feed/forage harvesting",Northern America,united states,USA,807.70,231.70,...,348.92,576,1384,1384,95,1,Valeurs Open Data de décembre 2016 maintenues ...,2017,1,1
3,2017-01-01,1,forage harvester (self propelled),agriculture,"cereals, cotton, & feed/forage harvesting",Northern America,canada,CAN,4.13,0.00,...,4.01,7,12,12,95,1,Valeurs Open Data de décembre 2016 maintenues ...,2017,1,1
4,2017-01-01,1,forage harvester (self propelled),agriculture,"cereals, cotton, & feed/forage harvesting",Northern America,united states,USA,5.52,0.52,...,2.46,5,11,11,95,1,Valeurs Open Data de décembre 2016 maintenues ...,2017,1,1


In [28]:
# Première et dernière date

date_min = min(
    fact_sales["date_mois"].min(),
    fact_predictions["date_mois"].min(),
    fact_stock_forecast["date_mois"].min()
)

date_max = max(
    fact_sales["date_mois"].max(),
    fact_predictions["date_mois"].max(),
    fact_stock_forecast["date_mois"].max()
)

# Calendrier mensuel

dim_date = pd.DataFrame({
    "date_mois": pd.date_range(
        start=date_min,
        end=date_max,
        freq="MS"
    )
})

# Variables temporelles

dim_date["date_id"] = (
    dim_date["date_mois"].dt.strftime("%Y%m")
    .astype(int)
)

dim_date["annee"] = (
    dim_date["date_mois"].dt.year
)

dim_date["mois"] = (
    dim_date["date_mois"].dt.month
)

dim_date["trimestre"] = (
    dim_date["date_mois"].dt.quarter
)

dim_date["annee_mois"] = (
    dim_date["date_mois"].dt.strftime("%Y-%m")
)

# Noms des mois en français

noms_mois = {
    1: "Janvier",
    2: "Février",
    3: "Mars",
    4: "Avril",
    5: "Mai",
    6: "Juin",
    7: "Juillet",
    8: "Août",
    9: "Septembre",
    10: "Octobre",
    11: "Novembre",
    12: "Décembre"
}

dim_date["nom_mois"] = (
    dim_date["mois"]
    .map(noms_mois)
)

dim_date = dim_date[
    [
        "date_id",
        "date_mois",
        "annee",
        "mois",
        "nom_mois",
        "trimestre",
        "annee_mois"
    ]
]

print("dim_date créée")
print("Dimensions :", dim_date.shape)
print("Première date :", dim_date["date_mois"].min())
print("Dernière date :", dim_date["date_mois"].max())

dim_date.head()

dim_date créée
Dimensions : (132, 7)
Première date : 2007-01-01 00:00:00
Dernière date : 2017-12-01 00:00:00


,date_id,date_mois,annee,mois,nom_mois,trimestre,annee_mois
0,200701,2007-01-01,2007,1,Janvier,1,2007-01
1,200702,2007-02-01,2007,2,Février,1,2007-02
2,200703,2007-03-01,2007,3,Mars,1,2007-03
3,200704,2007-04-01,2007,4,Avril,2,2007-04
4,200705,2007-05-01,2007,5,Mai,2,2007-05


In [29]:
# Dimension produit

dim_product = (
    dataset_final[
        [
            "produit",
            "categorie_produit_1",
            "categorie_produit_2"
        ]
    ]
    .drop_duplicates(
        subset=["produit"]
    )
    .sort_values("produit")
    .reset_index(drop=True)
)

# Identifiant produit

dim_product["product_id"] = (
    range(
        1,
        len(dim_product) + 1
    )
)

dim_product = dim_product[
    [
        "product_id",
        "produit",
        "categorie_produit_1",
        "categorie_produit_2"
    ]
]

# Ajout de l'identifiant dans les tables de faits

fact_sales = fact_sales.merge(
    dim_product[
        [
            "product_id",
            "produit"
        ]
    ],
    on="produit",
    how="left"
)

fact_predictions = fact_predictions.merge(
    dim_product[
        [
            "product_id",
            "produit"
        ]
    ],
    on="produit",
    how="left"
)

fact_stock_forecast = fact_stock_forecast.merge(
    dim_product[
        [
            "product_id",
            "produit"
        ]
    ],
    on="produit",
    how="left"
)

print("dim_product créée")
print("Nombre de produits :", len(dim_product))

dim_product.head()

dim_product créée
Nombre de produits : 32


,product_id,produit,categorie_produit_1,categorie_produit_2
0,1,4 wheels tractor,agriculture,tractors & trailers
1,2,aerial cableway,agriculture,forestry
2,3,agricultural trailer,agriculture,tractors & trailers
3,4,cattle feeder mixer (self propelled),agriculture,livestock
4,5,cattle feeder mixer (trailed),agriculture,livestock


In [30]:
# Dimension géographique

dim_region = (
    dataset_final[
        [
            "code_pays",
            "pays",
            "region"
        ]
    ]
    .drop_duplicates(
        subset=["code_pays"]
    )
    .sort_values("code_pays")
    .reset_index(drop=True)
)

# Identifiant géographique

dim_region["region_id"] = (
    range(
        1,
        len(dim_region) + 1
    )
)

dim_region = dim_region[
    [
        "region_id",
        "code_pays",
        "pays",
        "region"
    ]
]

# Ajout de l'identifiant géographique

fact_sales = fact_sales.merge(
    dim_region[
        [
            "region_id",
            "code_pays"
        ]
    ],
    on="code_pays",
    how="left"
)

fact_predictions = fact_predictions.merge(
    dim_region[
        [
            "region_id",
            "code_pays"
        ]
    ],
    on="code_pays",
    how="left"
)

fact_stock_forecast = fact_stock_forecast.merge(
    dim_region[
        [
            "region_id",
            "code_pays"
        ]
    ],
    on="code_pays",
    how="left"
)

# Ajout de la clé date

fact_sales["date_id"] = (
    fact_sales["date_mois"]
    .dt.strftime("%Y%m")
    .astype(int)
)

fact_predictions["date_id"] = (
    fact_predictions["date_mois"]
    .dt.strftime("%Y%m")
    .astype(int)
)

fact_stock_forecast["date_id"] = (
    fact_stock_forecast["date_mois"]
    .dt.strftime("%Y%m")
    .astype(int)
)

print("dim_region créée")
print("Nombre de pays :", len(dim_region))

dim_region

dim_region créée
Nombre de pays : 2


,region_id,code_pays,pays,region
0,1,CAN,canada,Northern America
1,2,USA,united states,Northern America


In [31]:
# Enregistrement des dimensions

dim_date.to_csv(
    f"{POWERBI_DATA_PATH}/dim_date.csv",
    index=False
)

dim_product.to_csv(
    f"{POWERBI_DATA_PATH}/dim_product.csv",
    index=False
)

dim_region.to_csv(
    f"{POWERBI_DATA_PATH}/dim_region.csv",
    index=False
)

# Enregistrement des tables de faits

fact_sales.to_csv(
    f"{POWERBI_DATA_PATH}/fact_sales.csv",
    index=False
)

fact_predictions.to_csv(
    f"{POWERBI_DATA_PATH}/fact_predictions.csv",
    index=False
)

fact_stock_forecast.to_csv(
    f"{POWERBI_DATA_PATH}/fact_stock_forecast.csv",
    index=False
)

# Enregistrement des résultats du modèle

model_results.to_csv(
    f"{POWERBI_DATA_PATH}/model_metrics.csv",
    index=False
)

feature_importance.to_csv(
    f"{POWERBI_DATA_PATH}/feature_importance.csv",
    index=False
)

evaluation_summary.to_csv(
    f"{POWERBI_DATA_PATH}/evaluation_summary.csv",
    index=False
)

error_by_product.to_csv(
    f"{POWERBI_DATA_PATH}/error_by_product.csv",
    index=False
)

error_by_region.to_csv(
    f"{POWERBI_DATA_PATH}/error_by_region.csv",
    index=False
)

error_by_country.to_csv(
    f"{POWERBI_DATA_PATH}/error_by_country.csv",
    index=False
)

error_by_month.to_csv(
    f"{POWERBI_DATA_PATH}/error_by_month.csv",
    index=False
)

print("Fichiers BI enregistrés")

Fichiers BI enregistrés


In [32]:
# Vérification du périmètre

assert set(
    fact_sales["categorie_produit_1"]
    .str.lower()
    .unique()
) == {"agriculture"}

assert set(
    fact_sales["region"].unique()
) == {"Northern America"}

assert set(
    fact_sales["code_pays"].unique()
) == {"USA", "CAN"}

assert len(dim_region) == 2

assert (
    fact_stock_forecast["date_mois"].nunique()
    == 12
)

assert len(fact_stock_forecast) == 288

assert (
    dim_date["date_mois"].max()
    == pd.Timestamp("2017-12-01")
)

# Fichiers attendus

expected_files = [
    f"{POWERBI_DATA_PATH}/dim_date.csv",
    f"{POWERBI_DATA_PATH}/dim_product.csv",
    f"{POWERBI_DATA_PATH}/dim_region.csv",
    f"{POWERBI_DATA_PATH}/fact_sales.csv",
    f"{POWERBI_DATA_PATH}/fact_predictions.csv",
    f"{POWERBI_DATA_PATH}/fact_stock_forecast.csv",
    f"{POWERBI_DATA_PATH}/model_metrics.csv",
    f"{POWERBI_DATA_PATH}/feature_importance.csv",
    f"{POWERBI_DATA_PATH}/evaluation_summary.csv",
    f"{POWERBI_DATA_PATH}/error_by_product.csv",
    f"{POWERBI_DATA_PATH}/error_by_region.csv",
    f"{POWERBI_DATA_PATH}/error_by_country.csv",
    f"{POWERBI_DATA_PATH}/error_by_month.csv"
]

for file in expected_files:
    print(file, os.path.exists(file))

print("\nVérifications terminées")
print("Dates :", len(dim_date))
print("Produits :", len(dim_product))
print("Pays :", len(dim_region))
print("Ventes :", len(fact_sales))
print("Prédictions de test :", len(fact_predictions))
print("Prévisions futures :", len(fact_stock_forecast))

../data/powerbi/dim_date.csv True
../data/powerbi/dim_product.csv True
../data/powerbi/dim_region.csv True
../data/powerbi/fact_sales.csv True
../data/powerbi/fact_predictions.csv True
../data/powerbi/fact_stock_forecast.csv True
../data/powerbi/model_metrics.csv True
../data/powerbi/feature_importance.csv True
../data/powerbi/evaluation_summary.csv True
../data/powerbi/error_by_product.csv True
../data/powerbi/error_by_region.csv True
../data/powerbi/error_by_country.csv True
../data/powerbi/error_by_month.csv True

Vérifications terminées
Dates : 132
Produits : 32
Pays : 2
Ventes : 1331
Prédictions de test : 485
Prévisions futures : 288


In [33]:
resume = f"""
Résumé de la préparation des données BI

Périmètre :
- Secteur : agriculture
- Région : Northern America
- Pays : USA et Canada

Tables créées :
- dim_date.csv : {len(dim_date)} lignes
- dim_product.csv : {len(dim_product)} lignes
- dim_region.csv : {len(dim_region)} lignes
- fact_sales.csv : {len(fact_sales)} lignes
- fact_predictions.csv : {len(fact_predictions)} lignes
- fact_stock_forecast.csv : {len(fact_stock_forecast)} lignes

Période historique :
{fact_sales["date_mois"].min()} au {fact_sales["date_mois"].max()}

Période de prévision :
{fact_stock_forecast["date_mois"].min()} au {fact_stock_forecast["date_mois"].max()}
"""

with open(
    f"{POWERBI_REPORT_PATH}/powerbi_resume.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(resume)

print(resume)


Résumé de la préparation des données BI

Périmètre :
- Secteur : agriculture
- Région : Northern America
- Pays : USA et Canada

Tables créées :
- dim_date.csv : 132 lignes
- dim_product.csv : 32 lignes
- dim_region.csv : 2 lignes
- fact_sales.csv : 1331 lignes
- fact_predictions.csv : 485 lignes
- fact_stock_forecast.csv : 288 lignes

Période historique :
2007-01-01 00:00:00 au 2016-12-01 00:00:00

Période de prévision :
2017-01-01 00:00:00 au 2017-12-01 00:00:00

